In [ ]:
# Install libraries
!pip install pandas scikit-learn requests

import pandas as pd
import requests
import string

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score

# Download dataset automatically

url = "https://raw.githubusercontent.com/justmarkham/pycon-2016-tutorial/master/data/sms.tsv"

response = requests.get(url)

with open("sms.tsv","wb") as f:
    f.write(response.content)

# Load dataset

df = pd.read_csv(
    "sms.tsv",
    sep="\t",
    names=["label","message"]
)

print("Dataset Loaded Successfully")

print(df.head())

# Convert labels

df["label"] = df["label"].map({
    "ham":0,
    "spam":1
})

# Clean text

def clean_text(text):

    text = text.lower()

    text = "".join(
        c for c in text
        if c not in string.punctuation
    )

    return text

df["message"] = df["message"].apply(clean_text)

# Split data

X_train, X_test, y_train, y_test = train_test_split(

    df["message"],

    df["label"],

    test_size=0.2,

    random_state=42

)

# TF-IDF

vectorizer = TfidfVectorizer(
    stop_words="english"
)

X_train = vectorizer.fit_transform(X_train)

X_test = vectorizer.transform(X_test)

# Train model

model = MultinomialNB()

model.fit(
    X_train,
    y_train
)

# Predict

y_pred = model.predict(X_test)

accuracy = accuracy_score(
    y_test,
    y_pred
)

print("\nAccuracy:",round(accuracy*100,2),"%")

# Test examples

messages = [

"Congratulations! You won ₹50000",

"Hi Harika, your class starts at 10 AM tomorrow",

"Claim your free iPhone now",

"Can we meet today?"

]

print("\nPredictions:\n")

for msg in messages:

    prediction = model.predict(

        vectorizer.transform([msg])

    )[0]

    result = "SPAM" if prediction == 1 else "NOT SPAM"

    print(msg,"--> ",result)

Dataset Loaded Successfully
  label                                            message
0   ham  Go until jurong point, crazy.. Available only ...
1   ham                      Ok lar... Joking wif u oni...
2  spam  Free entry in 2 a wkly comp to win FA Cup fina...
3   ham  U dun say so early hor... U c already then say...
4   ham  Nah I don't think he goes to usf, he lives aro...

Accuracy: 96.86 %

Predictions:

Congratulations! You won ₹50000 -->  SPAM
Hi Harika, your class starts at 10 AM tomorrow -->  NOT SPAM
Claim your free iPhone now -->  SPAM
Can we meet today? -->  NOT SPAM
